# Read Data from Bronze Layer

This notebook connects to MinIO via the Iceberg catalog and reads ongoing BusWayPoint data 
that has been written to `iceberg.bus_bronze.bus_way_points`.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, min, max

### Initialize Spark Session
Configure Spark to connect to Iceberg REST Catalog and MinIO.

In [2]:
spark = SparkSession.builder \
    .appName("ReadBronzeData-Notebook") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "rest") \
    .config("spark.sql.catalog.iceberg.uri", "http://gravitino:9001/api/metalakes/default/catalogs/iceberg") \
    .config("spark.sql.catalog.iceberg.warehouse", "s3a://lakehouse/") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session created successfully!")

:: loading settings :: url = jar:file:/root/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7be0798e-c24e-4746-90c9-6e30b3bdac35;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.5 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.5 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 388ms :: artifacts dl 23ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.0 fr

Spark Session created successfully!


### Read Iceberg Table (Bronze Layer)
Load the table previously written.

In [4]:
table_name = "catalog_iceberg.bus_bronze.bus_way_points"
df = spark.read.format("iceberg").load(table_name)

print(f"Total records in {table_name}: {df.count()}")
df.printSchema()
df.show(5, truncate=False)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


Total records in catalog_iceberg.bus_bronze.bus_way_points: 214016
root
 |-- msgType: string (nullable = true)
 |-- vehicle: string (nullable = true)
 |-- driver: string (nullable = true)
 |-- speed: float (nullable = true)
 |-- datetime: integer (nullable = true)
 |-- x: double (nullable = true)
 |-- y: double (nullable = true)
 |-- z: float (nullable = true)
 |-- heading: float (nullable = true)
 |-- ignition: boolean (nullable = true)
 |-- aircon: boolean (nullable = true)
 |-- door_up: boolean (nullable = true)
 |-- door_down: boolean (nullable = true)
 |-- sos: boolean (nullable = true)
 |-- working: boolean (nullable = true)
 |-- analog1: float (nullable = true)
 |-- analog2: float (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- date: date (nullable = true)
 |-- load_at: timestamp (nullable = true)



+-------------------+----------------------------------------------------------------+----------------------------------------------------------------+-----+----------+----------+---------+----+-------+--------+------+-------+---------+----+-------+-------+-------+-------------------+----------+--------------------------+
|msgType            |vehicle                                                         |driver                                                          |speed|datetime  |x         |y        |z   |heading|ignition|aircon|door_up|door_down|sos |working|analog1|analog2|timestamp          |date      |load_at                   |
+-------------------+----------------------------------------------------------------+----------------------------------------------------------------+-----+----------+----------+---------+----+-------+--------+------+-------+---------+----+-------+-------+-------+-------------------+----------+--------------------------+
|MsgType_BusWayPoint|e738d84

### Basic Analysis
Summary statistics or aggregation to verify data integrity.

In [5]:
print("Records count by partition date:")
df.groupBy("date").count().orderBy("date").show()

print("Top 5 fastest speeds recorded:")
df.select("vehicle", "driver", "speed", "timestamp").orderBy(col("speed").desc()).show(5, truncate=False)

Records count by partition date:


+----------+------+
|      date| count|
+----------+------+
|2025-03-23|214016|
+----------+------+

Top 5 fastest speeds recorded:
+----------------------------------------------------------------+----------------------------------------------------------------+-----+-------------------+
|vehicle                                                         |driver                                                          |speed|timestamp          |
+----------------------------------------------------------------+----------------------------------------------------------------+-----+-------------------+
|d0e74b5a90d462d4b5391f3eee77b108c443ebd214bea369d8f2131a39679cb8|NULL                                                            |82.0 |2025-03-23 07:26:59|
|5f167b330be3da64ffd55e6f844a2071d1e6cf0b1469ab1bcab979add01208aa|88917658b5bff2e1b39bad7d467543609a050fd86e02bd71708230d0f7a24769|73.0 |2025-03-23 07:06:03|
|d0e74b5a90d462d4b5391f3eee77b108c443ebd214bea369d8f2131a39679cb8|NULL        

### Use Spark SQL Query
You can also query the table directly using SQL syntax via `spark.sql`.

In [6]:
spark.sql(f"""
    SELECT msgType, COUNT(*) as activity_count
    FROM {table_name}
    GROUP BY msgType
""").show()

+-------------------+--------------+
|            msgType|activity_count|
+-------------------+--------------+
|MsgType_BusWayPoint|        214016|
+-------------------+--------------+

